# Cell-Cell Interaction Analysis with Visium HD Data

This tutorial runs through stLearn CCI analysis on extremely large data, containing >583,000 8μm binned squares (when DEMO = False) with gene expression measured in space.

### Environment setup

In [ ]:
import os
import platform
import random

# Make sure all the seeds are set
seed = 0
random.seed(seed)

# Only constrain threads on macOS where BLAS/numba deadlocks are common.
# Must run before any numpy/scanpy import.
if platform.system() == "Darwin":
    os.environ["OPENBLAS_NUM_THREADS"] = "1"
    os.environ["MKL_NUM_THREADS"] = "1"
    os.environ["NUMBA_NUM_THREADS"] = "1"
    n_cpus = 1
else:
    n_cpus = None

In [ ]:
import stlearn
import pathlib

import matplotlib.pyplot as plt
import numpy as np

stlearn.settings.set_figure_params(dpi=120)
np.random.seed(seed)

# Ignore all warnings
import warnings

warnings.filterwarnings("ignore")

### Loading the data
For the demo the sample is cropped to its centre quarter (`DEMO = True`); set it to `False` for the whole sample. The bins are filtered, normalised and clustered with Leiden; the clusters stand in for cell types below.

In [ ]:
# Setup download directory and get data
stlearn.settings.datasetdir = pathlib.Path.cwd().parent / "data"
sample_id = "Visium_HD_FF_Human_Breast_Cancer"
bin_size = 8
table_name = f"square_{bin_size:03d}um"
data_dir = stlearn.settings.datasetdir / sample_id
DEMO = True

In [ ]:
# Crop the sample to the center quarter using a bounding box or create a checkpoint with the whole thing
def full_sample():
    return stlearn.datasets.visium_hd_sge(sample_id=sample_id, bin_size=bin_size)

if DEMO:
    sdata = stlearn.io.store_checkpoint(
        data_dir / f"{sample_id}_{table_name}_demo.zarr",
        lambda: stlearn.io.crop_visium_hd(full_sample(), sample_id, table_name),
    )
else:
    sdata = stlearn.io.store_checkpoint(
        data_dir / f"{sample_id}_{table_name}.zarr",
        lambda: full_sample()
    )

adata = sdata.tables[table_name]
print(f"{adata.n_obs:,} bins x {adata.n_vars:,} genes")

In [ ]:
# QC - Filter genes and cells with at least 10 counts
clustered_name = f"{table_name}_clustered"

def cluster():
    adata = sdata.tables[table_name].copy()
    stlearn.pp.filter_genes(adata, min_counts=10)
    stlearn.pp.filter_cells(adata, min_counts=10)
    adata.layers["counts"] = adata.X.copy()
    stlearn.pp.normalize_total(adata)
    stlearn.pp.log1p(adata)
    stlearn.pp.highly_variable_genes(adata, n_top_genes=2000)
    stlearn.em.run_pca(adata, n_comps=50, random_state=seed)
    stlearn.pp.neighbors(adata, n_neighbors=25, use_rep="X_pca", random_state=seed)
    stlearn.tl.clustering.leiden(adata, resolution=1.05, random_state=seed)
    return adata

adata = stlearn.io.checkpoint(sdata, clustered_name, cluster)

In [ ]:
# Add backwards compatibility - image and spatial data.
adata = stlearn.io.attach_spatial(adata, data_dir / "binned_outputs" / table_name / "spatial", library_id=sample_id)

In [ ]:
stlearn.pl.sdata_plot(sdata, "leiden", table_key=clustered_name)

In [ ]:
sdata.pl.render_images("Visium_HD_FF_Human_Breast_Cancer_lowres_image").pl.show("Visium_HD_FF_Human_Breast_Cancer")

In [ ]:
# Copy raw counts back - these are to be gridded.
adata.X = adata.layers["counts"].copy()

## Gridding

`st.tl.cci.run` scores each unit for an LR pair and compares the score with 1,000 random gene pairs of similar abundance scored at the same unit. That comparison needs units in which most genes are detected, as a Visium spot (several cells) is and an 8µm bin (about one cell) is not: at that resolution almost no random pair is co-detected at a bin, so any co-detection comes out significant whatever the biology. The bins are therefore summed into squares first.

The square size sets how many bins a unit holds. 30 µm squares hold about 14 bins (a Visium spot) following the design of the test. Smaller squares approach a single bin, and much larger ones blur the tissue structure (check the plots below). A square's counts are the sum of its bins' raw counts; the squares are then normalised as Visium spots would be.

In [ ]:
# Grid squares of about 30 µm.
grid_um = 30
n_col, n_row = stlearn.utils.get_shape(adata, bin_um=grid_um)
print(f'{n_row} by {n_col} has this many spots:\n', n_row * n_col)

By providing 'use_label' to the function below, the share of each Leiden cluster in every square is saved in grid.uns['leiden']. That way we can perform stLearn CCI with the cell type information!

In [ ]:
### Gridding.
grid_name = f"grid_{n_row}x{n_col}"

def make_grid():
    grid = stlearn.tl.cci.create_grid(adata, bin_um=grid_um, use_label='leiden')
    grid.layers["counts"] = grid.X.copy()
    stlearn.pp.normalize_total(grid)
    return grid

grid = stlearn.io.checkpoint(sdata, grid_name, make_grid)
print(f"{grid.n_obs:,} squares with bins; bins per square: median {grid.obs['n_cells'].median():.0f}")

### Checking the gridding

Comparing the gridded data to the original, to make sure it makes sense.

It's recommended to visualise the dominant cell types per spot, in order to gauge whether the tissue structure is adequately maintained after gridding (i.e. to make sure it is not too low resolution!).

In [ ]:
fig, axes = plt.subplots(ncols=2, figsize=(20, 5))
stlearn.pl.cluster_plot(grid, use_label='leiden', size=10, ax=axes[0], show_plot=False, show_image=False)
stlearn.pl.cluster_plot(adata, use_label='leiden', ax=axes[1], show_plot=False, show_image=False, size=2)
axes[0].set_title(f'Grid leiden dominant spots')
axes[0].set_aspect('equal')
axes[1].set_title(f'Bins leiden labels')
axes[1].set_aspect('equal')
plt.show()

In [ ]:
# Clusters 5/7/3 matched to immune/dcis/stroma by markers for this data and seed.
groups = list(grid.obs['leiden'].cat.categories)
immune_rich_cluster = groups[5]
dcis_cluster = groups[7]
stroma_cluster = groups[3]
for group in [immune_rich_cluster, dcis_cluster, stroma_cluster]:
    fig, axes = plt.subplots(ncols=3, figsize=(20, 5))
    grid.obs['group'] = grid.uns['leiden'][group].values
    stlearn.pl.feat_plot(
        grid, feature='group', ax=axes[0], show_plot=False, vmax=1, show_color_bar=False,
        show_image=False
    )
    stlearn.pl.cluster_plot(
        grid, use_label='leiden', list_clusters=[group], ax=axes[1], show_plot=False,
        show_image=False
    )
    stlearn.pl.cluster_plot(
        adata, use_label='leiden', list_clusters=[group], ax=axes[2], show_plot=False,
        show_image=False
    )
    axes[0].set_title(f'Grid {group} proportions (max = 1)')
    axes[0].set_aspect('equal')
    axes[1].set_title(f'Grid {group} max spots')
    axes[1].set_aspect('equal')
    axes[2].set_title(f'Individual bin {group}')
    axes[2].set_aspect('equal')
    plt.show()

In [ ]:
gene_name = 'CD74'

x = adata.obs_vector(gene_name)
vmax = np.percentile(x[x > 0], 99)

fig, axes = plt.subplots(ncols=2, figsize=(20, 8))
stlearn.pl.gene_plot(grid, gene_symbols=gene_name, ax=axes[0], show_color_bar=False, show_plot=False, show_image=False)
stlearn.pl.gene_plot(
    adata, gene_symbols=gene_name, ax=axes[1], show_color_bar=False, show_plot=False, vmax=vmax, show_image=False
)
axes[0].set_aspect('equal')
axes[1].set_aspect('equal')
axes[0].set_title(f'Grid {gene_name} expression')
axes[1].set_title(f'Binned {gene_name} expression')
plt.show()

## LR Permutation Test

Running the LR permutation test, to determine regions of high LR co-expression.

In [ ]:
# Loading the LR databases available within stlearn.
lrs = stlearn.tl.cci.lr_pairs('connectomeDB2025', species='human')
print(len(lrs))

In [ ]:
# adjacent squares, diagonals included (8 neighbours)
distance_in_um = 1.5 * grid_um
distance = stlearn.utils.microns_to_pixels(grid, distance_in_um)

# Running the analysis #
lr_name = f"{grid_name}_lr_{distance_in_um:g}um"
lr_results = stlearn.io.checkpoint(
    sdata, lr_name, compute=lambda:
    stlearn.tl.cci.run(
        grid,
        lrs,
        min_spots=20,       # Drop LR pairs scored in 20 or fewer spots
        distance=distance,  # Radius of each square's neighbourhood
        n_pairs=1000,       # Random gene pairs for the background; ~10,000 for real results
        n_cpus=n_cpus,      # None uses every CPU
        random_state=seed,
    )
)

# Added the neighbours to grid
stlearn.io.save_table(sdata, grid_name)

In [ ]:
# A dataframe detailing the LR pairs ranked by number of significant spots.
lr_info = lr_results.var
# Take the first LR
interesting_lr = lr_results.var.index[0]
lr_info

In [ ]:
# Showing the rankings of the LR from a global and local perspective.
# Ranking based on number of significant hotspots.
stlearn.pl.lr_summary(lr_results, n_top=500)
stlearn.pl.lr_summary(lr_results, n_top=50, figsize=(10, 3))

### Downstream visualisations from LR analysis

For more downstream visualisations, please see the stLearn CCI tutorial:

https://stlearn.readthedocs.io/en/latest/tutorials/cell_cell_interaction.html

In [ ]:
stats = stlearn.tl.cci.LR_RESULTS
fig, axes = plt.subplots(ncols=len(stats), figsize=(20, 8))
for i, stat in enumerate(stats):
    stlearn.pl.lr_result_plot(
        grid, lr_results, use_result=stat, use_lr=interesting_lr, show_color_bar=False, ax=axes[i], show_image=False
    )
    axes[i].set_aspect('equal')
    axes[i].set_title(f'{interesting_lr} {stat}')

## Predicting Significant Cell-Cell Interactions

For this analysis, we are using each square’s cluster proportions, stored in grid.uns['leiden'], as automatically determined by stlearn.tl.cci.grid. The permutation is performed to permute the cell proportions associated with each spot, to determine spots located in LR hotspots that have certain cell types over-represented.

In [ ]:
stlearn.tl.cci.run_cci(
    grid,
    'leiden',              # Cell information either in data.obs and data.uns
    lr_results,
    min_spots=2,           # Minimum number of spots for LR to be tested.
    spot_mixtures=True,    # Count every cluster in a square, not just its main one
    cell_prop_cutoff=0.2,  # A cluster counts if it is more than 20% of the square's bins
    sig_spots=True,        # Only consider neighbourhoods of spots which had significant LR scores.
    n_perms=100,           # Permutations of cells to get background, recommend ~1000
    random_state=seed,
    n_cpus=n_cpus,
)
# TODO - need to refactor results and plotting to load/store efficient.
# lr_results = stlearn.io.checkpoint(sdata, lr_name, run_cci_run, done=lambda t: "lr_cci_leiden" in t.uns)

## Diagnostic: checking for interaction and cell type frequency correlation

Should be little to no correlation; indicating the permutation has adequately controlled for cell type frequency.

In [ ]:
stlearn.pl.cci_check(grid, 'leiden', lr_results, figsize=(16, 5))

## CCI Visualisations

### CCI Network Plot

In [ ]:
pos_1 = stlearn.pl.ccinet_plot(grid, 'leiden', lr_results, return_pos=True, min_counts=30)
stlearn.pl.ccinet_plot(
    grid, 'leiden', lr_results, interesting_lr, min_counts=2,
    figsize=(10, 7.5), pos=pos_1
)

### CCI Chord Plot

In [ ]:
stlearn.pl.lr_chord_plot(grid, 'leiden', lr_results)
stlearn.pl.lr_chord_plot(grid, 'leiden', lr_results, interesting_lr)

## Acknowledgements

Thanks to Dr Pete Simpsons (https://about.uq.edu.au/experts/1466) who annotated the H&E to help categorise the morphology.